In [ ]:
# Ejemplo numérico: SGD vs Momentum vs Adam (toy)
import numpy as np

def g(theta):
    return 2*theta

theta0 = 1.0

# SGD
eta_sgd = 0.1
theta_sgd = theta0 - eta_sgd * g(theta0)

# Momentum (dos pasos)
gamma = 0.9
v0 = 0.0
v1 = gamma*v0 + eta_sgd * g(theta0)
theta_mom1 = theta0 - v1
v2 = gamma*v1 + eta_sgd * g(theta_mom1)
theta_mom2 = theta_mom1 - v2

# Adam (primer paso aproximado)
eta_adam = 0.01
beta1 = 0.9
beta2 = 0.999
eps = 1e-8
m1 = (1-beta1)*g(theta0)
v1_adam = (1-beta2)*(g(theta0)**2)
# correcciones de sesgo (primer paso)
mhat = m1/(1-beta1)
vhat = v1_adam/(1-beta2)
delta = eta_adam * mhat / (np.sqrt(vhat) + eps)
theta_adam = theta0 - delta

print('theta0 =', theta0)
print('SGD step ->', theta_sgd)
print('Momentum step1 ->', theta_mom1, ' step2 ->', theta_mom2)
print('Adam approx step ->', theta_adam)


### Ejemplo numérico rápido

Mismo ejemplo corto que en el documento teórico: actualizaciones de un parámetro en $L(\theta)=\theta^2$ usando SGD, momentum y Adam (primer paso). Ejecuta la celda siguiente para ver los números.

In [ ]:
# Celda opcional para Colab: instalar torchvision si falta y mostrar info de CUDA
import importlib, sys, subprocess
missing = []
for p in ("torch","torchvision"):
    if importlib.util.find_spec(p) is None:
        missing.append(p)
if missing:
    print('Instalando paquetes:', missing)
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q'] + missing)
else:
    import torch
    print('torch', torch.__version__)
    print('CUDA available:', torch.cuda.is_available())
    if torch.cuda.is_available():
        try:
            print('Device name:', torch.cuda.get_device_name(0))
        except Exception:
            pass

# Montar Drive (descomentar en Colab):
# from google.colab import drive
# drive.mount('/content/drive')


## Colab — instrucciones rápidas

- Para ejecutar en Google Colab: sube este notebook en https://colab.research.google.com/ (Upload) y activa Runtime → Change runtime type → GPU.
- Si quieres persistir resultados monta Google Drive (celda de ejemplo más abajo).
- Si faltan paquetes (p. ej. `torchvision`), ejecuta la celda de instalación a continuación.


# Práctico: Optimizadores en PyTorch

[Open in Colab](https://colab.research.google.com/) — sube este notebook a Colab o ábrelo desde Drive.

Comparativa práctica de `torch.optim` (SGD, SGD+momentum, RMSprop, Adam, AdamW) sobre Fashion‑MNIST.
Objetivo: comparar curvas de pérdida y accuracy, y mostrar uso de schedulers.

## Instrucciones
- Ejecuta las celdas en orden.
- En Colab: activa GPU (Runtime → Change runtime type → GPU).
- Para pruebas rápidas: reduce `EPOCHS` o `BATCH_SIZE`.

In [ ]:
# Comprobación rápida para Colab / GPU
import torch
print('torch', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    try:
        print('Device name:', torch.cuda.get_device_name(0))
    except Exception:
        print('Device name: (no disponible)')

# En Colab: para guardar resultados monta Drive si quieres persistencia:
# from google.colab import drive
# drive.mount('/content/drive')

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import matplotlib.pyplot as plt
import numpy as np
print('torch', torch.__version__)


In [ ]:
# Datos: FashionMNIST con transform a tensor y normalización simple
transform = transforms.Compose([transforms.ToTensor()])
train_ds = datasets.FashionMNIST(root='data', train=True, download=True, transform=transform)
test_ds = datasets.FashionMNIST(root='data', train=False, download=True, transform=transform)
print('Train', len(train_ds), 'Test', len(test_ds))


In [ ]:
class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 32, 3, padding=1)
        self.pool = nn.MaxPool2d(2)
        self.fc1 = nn.Linear(32*14*14, 128)
        self.fc2 = nn.Linear(128, 10)
    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = self.pool(x)
        x = x.view(x.size(0), -1)
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x


In [ ]:
EPOCHS = 3
BATCH_SIZE = 256
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

def train_one_epoch(model, loader, loss_fn, opt):
    model.train()
    total_loss = 0.0
    correct = 0
    total = 0
    for X, y in loader:
        X, y = X.to(DEVICE), y.to(DEVICE)
        opt.zero_grad()
        out = model(X)
        loss = loss_fn(out, y)
        loss.backward()
        opt.step()
        total_loss += loss.item() * X.size(0)
        preds = out.argmax(dim=1)
        correct += (preds == y).sum().item()
        total += X.size(0)
    return total_loss/total, correct/total

def eval_model(model, loader, loss_fn):
    model.eval()
    total_loss = 0.0
    correct = 0
    total = 0
    with torch.no_grad():
        for X, y in loader:
            X, y = X.to(DEVICE), y.to(DEVICE)
            out = model(X)
            loss = loss_fn(out, y)
            total_loss += loss.item() * X.size(0)
            preds = out.argmax(dim=1)
            correct += (preds == y).sum().item()
            total += X.size(0)
    return total_loss/total, correct/total


In [ ]:
# Ejecutar experimentos con distintos optimizadores
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE)
loss_fn = nn.CrossEntropyLoss()

optim_set = [
    ('SGD', optim.SGD, {'lr':0.01}),
    ('SGD+momentum', optim.SGD, {'lr':0.01, 'momentum':0.9}),
    ('RMSprop', optim.RMSprop, {'lr':0.001}),
    ('Adam', optim.Adam, {'lr':0.001}),
    ('AdamW', optim.AdamW, {'lr':0.001, 'weight_decay':1e-4}),
]

results = {}
for name, OptClass, opt_kwargs in optim_set:
    print('Running', name)
    model = SimpleCNN().to(DEVICE)
    opt = OptClass(model.parameters(), **opt_kwargs)
    history = {'train_loss':[], 'train_acc':[], 'val_loss':[], 'val_acc':[]}
    for epoch in range(EPOCHS):
        tl, ta = train_one_epoch(model, train_loader, loss_fn, opt)
        vl, va = eval_model(model, test_loader, loss_fn)
        history['train_loss'].append(tl)
        history['train_acc'].append(ta)
        history['val_loss'].append(vl)
        history['val_acc'].append(va)
        print(f'Epoch {epoch+1}/{EPOCHS}  train_loss={tl:.4f} train_acc={ta:.4f}  val_loss={vl:.4f} val_acc={va:.4f}')
    results[name] = history

# Plots: pérdida y accuracy de validación
plt.figure(figsize=(10,4))
for name,h in results.items():
    plt.plot(h['val_loss'], label=name)
plt.title('Validation loss comparison')
plt.legend()
plt.show()

plt.figure(figsize=(10,4))
for name,h in results.items():
    plt.plot(h['val_acc'], label=name)
plt.title('Validation accuracy comparison')
plt.legend()
plt.show()


## Extensiones sugeridas
- Añadir schedulers: `optim.lr_scheduler.StepLR`, `CosineAnnealingLR`, `OneCycleLR`.
- Comparar con `weight_decay` vs `L2` en Adam/AdamW.
- Ejecutar con distinto `batch_size` y `learning_rate` para ver sensibilidad.